# Profiling API — Testes

Testa todos os endpoints de `/profiling`:
- `POST /profiling/ingest` — Ingestão + predição (CSV, background)
- `GET /profiling/vehicle/{veiculo_id}?target=` — Info de veículo

In [1]:
import httpx
import pandas as pd
from pathlib import Path

BASE_URL = "http://localhost:8010"
client = httpx.Client(base_url=BASE_URL, timeout=120)

In [2]:
TARGET = "h"
DATA_PATH = '../../../datasets/full/telemetria_movias2025_features.csv'

In [10]:
import sys
import os

sys.path.append(os.path.abspath('../..'))

from moviasai.data.utils import load_raw_data

import polars as pl

from pathlib import Path
from datetime import date

df = load_raw_data(DATA_PATH)

out_dir = Path("../../../tmp/segments")
out_dir.mkdir(parents=True, exist_ok=True)

# Garantir coluna data como Date
df_seg = df.with_columns(pl.col("data").cast(pl.Date))

# Cutoffs: dividir em três segmentos
cutoff1 = date(2025, 8, 24)
cutoff2 = date(2025, 9, 28)

seg1 = df_seg.filter(pl.col("data") <= cutoff1)
seg2 = df_seg.filter((pl.col("data") > cutoff1) & (pl.col("data") <= cutoff2))
seg3 = df_seg.filter(pl.col("data") > cutoff2)

segments = [seg1, seg2, seg3]

segment_paths = []
for i, seg in enumerate(segments, start=1):
    path = out_dir / f"segment_{i}.csv"
    seg.write_csv(path)
    segment_paths.append(path)
    print(f"segment_{i}.csv: {seg.shape[0]} linhas, {seg['data'].min()} a {seg['data'].max()}")

print(f"\nTotal: {len(segments)} segmentos")

segment_1.csv: 4759884 linhas, 2025-01-01 a 2025-08-24
segment_2.csv: 705915 linhas, 2025-08-25 a 2025-09-28
segment_3.csv: 242028 linhas, 2025-09-29 a 2025-10-10

Total: 3 segmentos


## 1. Ingestão + Predição via CSV

Envia CSV com colunas `veiculo_id, data, h_dia_clean, km_dia_clean`.
Executa ingestão de perfis e predição para ambos os targets (h, km) numa transação atómica.

In [11]:
def run_ingestion(segment_paths, idx=0):
    csv_path = segment_paths[idx]

    if csv_path.exists():
        with open(csv_path, "rb") as f:
            resp = client.post("/profiling/ingest", files={"file": (csv_path.name, f, "text/csv")})
        print(f"Status: {resp.status_code}")
    else:
        print(f"CSV não encontrado: {csv_path}")
    return resp

In [16]:
resp = run_ingestion(segment_paths, idx=2)
resp.json()

Status: 202


{'run_id': 42, 'message': 'Ingestão e predição submetidas em background.'}

In [17]:

# Listar todos os runs de ingestão
import pandas as pd

resp_all = client.get("/pipeline/runs", params={"step": "ingestion"})
print(f"Status: {resp_all.status_code}")
pd.DataFrame(resp_all.json())


Status: 200


,id,step,target,model_type,status,started_at,finished_at,error_message,metrics,artifacts,created_at
0,42,ingestion,global,None,completed,2026-04-30T12:21:10.964644-03:00,2026-04-30T12:28:50.200301-03:00,None,{'ingestion': {'daily_activity_inserted': 8253...,None,2026-04-30T12:21:10-03:00
1,41,ingestion,global,None,completed,2026-04-30T11:40:07.595816-03:00,2026-04-30T12:17:46.567360-03:00,None,{'ingestion': {'daily_activity_inserted': 2302...,None,2026-04-30T11:40:07-03:00
2,40,ingestion,global,None,completed,2026-04-30T11:33:58.385874-03:00,2026-04-30T11:35:56.072158-03:00,None,{'ingestion': {'daily_activity_inserted': 1303...,None,2026-04-30T11:33:58-03:00
3,39,ingestion,global,None,running,2026-04-30T10:59:40.133897-03:00,None,None,None,None,2026-04-30T10:59:40-03:00
4,38,ingestion,global,None,completed,2026-04-30T10:39:00.116534-03:00,2026-04-30T10:42:38.875483-03:00,None,{'ingestion': {'daily_activity_inserted': 1303...,None,2026-04-30T10:39:00-03:00
5,37,ingestion,global,None,completed,2026-04-30T09:35:15.938609-03:00,2026-04-30T09:45:32.896733-03:00,None,{'ingestion': {'daily_activity_inserted': 3128...,None,2026-04-30T09:35:15-03:00
6,36,ingestion,global,None,completed,2026-04-30T09:32:12.322840-03:00,2026-04-30T09:34:14.871557-03:00,None,{'ingestion': {'daily_activity_inserted': 1303...,None,2026-04-30T09:32:12-03:00
7,35,ingestion,global,None,running,2026-04-29T14:29:06.537659-03:00,None,None,None,None,2026-04-29T14:29:06-03:00
8,34,ingestion,global,None,failed,2026-04-29T14:15:46.490168-03:00,2026-04-29T14:21:09.604286-03:00,"Traceback (most recent call last):\n File ""C:...",None,None,2026-04-29T14:15:46-03:00
9,33,ingestion,global,None,completed,2026-04-29T14:12:08.224526-03:00,2026-04-29T14:14:16.060961-03:00,None,{'ingestion': {'daily_activity_inserted': 1303...,None,2026-04-29T14:12:08-03:00


## 2. Consultar informações de um veículo

Retorna perfil (features) e metadados para um veículo e target.

In [51]:
VEICULO_ID = 4
TARGET = "km"

resp = client.get(f"/profiling/vehicle/{VEICULO_ID}", params={"target": TARGET})
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'veiculo_id': 4,
 'target': 'km',
 'profile': {'cluster_0_km': 0.9999881982803345,
  'cluster_1_km': 1.1812920092779677e-05,
  'cluster_2_km': 1.931314830283526e-11,
  'cycle_mean_fim_km': 11.533333333333337,
  'cycle_mean_inicio_km': 19.209259259259284,
  'cycle_mean_meio_km': 17.626851851851868,
  'cycle_prob_active_fim_km': 0.6,
  'cycle_prob_active_inicio_km': 0.7592592592592593,
  'cycle_prob_active_meio_km': 0.7962962962962963,
  'cycle_ratio_fim_inicio_km': 0.6004048973296051,
  'day_1_cv_km': 0.716382436202102,
  'day_1_iqr_km': 11.400000000000093,
  'day_1_mean_km': 18.939583333333445,
  'day_1_p25_km': 13.400000000000093,
  'day_1_p75_km': 24.800000000000185,
  'day_1_prob_active_km': 0.9583333333333334,
  'day_1_std_km': 13.567984848986141,
  'day_2_cv_km': 0.8967878593943882,
  'day_2_iqr_km': 16.199999999999932,
  'day_2_mean_km': 20.599999999999902,
  'day_2_p25_km': 8.400000000000091,
  'day_2_p75_km': 24.600000000000023,
  'day_2_prob_active_km': 0.9166666666666666,
  

In [45]:
# Testar com target 'h'
resp = client.get(f"/profiling/vehicle/{VEICULO_ID}", params={"target": "h"})
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'veiculo_id': 4,
 'target': 'h',
 'profile': {'cluster_0_h': 6.735324859619141e-06,
  'cluster_1_h': 0.9999932646751404,
  'cycle_mean_fim_h': 2.326783564814815,
  'cycle_mean_inicio_h': 2.5434207818930044,
  'cycle_mean_meio_h': 2.6661471193415642,
  'cycle_prob_active_fim_h': 0.75,
  'cycle_prob_active_inicio_h': 0.7962962962962963,
  'cycle_prob_active_meio_h': 0.7592592592592593,
  'cycle_ratio_fim_inicio_h': 0.9148244684401173,
  'day_1_cv_h': 0.824592387684745,
  'day_1_iqr_h': 5.267430555555556,
  'day_1_mean_h': 2.8713107638888893,
  'day_1_p25_h': 1.065,
  'day_1_p75_h': 6.332430555555557,
  'day_1_prob_active_h': 0.9583333333333334,
  'day_1_std_h': 2.3676609985800483,
  'day_2_cv_h': 0.819243752045211,
  'day_2_iqr_h': 5.434652777777779,
  'day_2_mean_h': 3.114652777777778,
  'day_2_p25_h': 0.8977777777777778,
  'day_2_p75_h': 6.332430555555557,
  'day_2_prob_active_h': 0.9166666666666666,
  'day_2_std_h': 2.551659827984706,
  'day_3_cv_h': 0.8938801045945081,
  'day_3_iqr_

In [52]:
# Testar veículo inexistente → deve retornar 404
resp = client.get("/profiling/vehicle/999999", params={"target": "km"})
print(f"Status: {resp.status_code}")
resp.json()

Status: 404


{'detail': 'Veículo 999999 não existe ou não teve atividade recente.'}

## 3. Profile Metadata

Histórico de metadados do perfil — cada ingestão cria um novo registo.
Usar `?last=true` para obter apenas o mais recente.

In [53]:
# Último profile metadata
resp = client.get("/profiling/metadata", params={"last": True})
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


[{'id': 2,
  'n_veiculos': 10378,
  'dt_inicio': '2025-03-17',
  'dt_fim': '2025-08-31',
  'sample_size': 168,
  'created_at': '2026-04-27 17:32:47'}]

In [3]:
# Histórico completo
resp = client.get("/profiling/metadata")
print(f"Status: {resp.status_code}")
pd.DataFrame(resp.json())

Status: 200


,id,n_veiculos,dt_inicio,dt_fim,sample_size,created_at
0,4,10441,2025-03-31,2025-09-14,168,2026-04-27 19:25:11
1,3,10422,2025-03-24,2025-09-07,168,2026-04-27 18:55:35
2,2,10378,2025-03-17,2025-08-31,168,2026-04-27 18:46:28
3,1,10357,2025-03-10,2025-08-24,168,2026-04-27 18:35:58
